# 03 · Evaluate (GPU)

Scores zero-shot, every DAPT adapter, and each task row on the TEST splits → `results/`. Verdict Macro-F1 + ROUGE always; set `use_judge=True` for the CLEV LLM-judge. The judge models come from `FEDDAPT_JUDGE_MODELS` — API (`claude-…`) or local (`ollama:…`), and must differ from the Mistral-7B base.

In [ ]:
# --- setup: run once per Colab session ---
import os
# 1) get the code (skip if you opened this notebook from the GitHub tab)
if not os.path.exists('pyproject.toml'):
    if not os.path.exists('fedapt'):
        get_ipython().system('git clone https://github.com/YOUR_USERNAME/fedapt.git')
    get_ipython().run_line_magic('cd', 'fedapt')
# 2) install (keep the extras — Colab has the GPU for [train]/[eval])
get_ipython().system('pip -q install -e ".[train,eval]"')
# 3) persist corpus/adapters/results to Google Drive so a dropped session resumes.
#    On Colab, Config auto-defaults FEDDAPT_ROOT to /content/drive/MyDrive/FedDAPT.
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass

In [ ]:
from fedapt.config import load_config
from fedapt import evaluate
cfg = load_config()

### Reference metrics only (no judge)

In [ ]:
evaluate.run_eval(cfg, use_judge=False)

### CLEV judge
The student's generate pass and the judge both want the GPU, so run this **after** the cell above frees the base model. For a local judge on Colab, run the Ollama cell first (pull two judges, e.g. `gemma3:12b` + `llama3.1:8b`) and set `FEDDAPT_JUDGE_MODELS=ollama:gemma3:12b,ollama:llama3.1:8b`.

In [ ]:
# --- OPTIONAL: run a strong LOCAL teacher/judge on Colab's GPU via Ollama ---
# The open-weight models (Qwen/Gemma/Llama) run on the SAME GPU as training, so
# only do this during data-build (nb 00) or the judge pass (nb 03) — never during
# training. T4: qwen3:14b / gemma3:12b.  A100: qwen3:32b / gemma3:27b.
get_ipython().system('curl -fsSL https://ollama.com/install.sh | sh')
import subprocess, time, os
subprocess.Popen(['ollama', 'serve'])          # background server
time.sleep(5)
get_ipython().system('ollama pull qwen3:14b')
os.environ['FEDDAPT_OLLAMA_HOST'] = 'http://localhost:11434'

In [ ]:
# evaluate.run_eval(cfg, use_judge=True)   # validate the judge first (nb-free: scripts/score_judge.py)

Next → **04 Analysis**.